# Train `cxr-temporal-multiprior` on Colab

This notebook runs **actual pretraining** on Colab using a MIMIC-CXR
subset zip you upload to Google Drive once.

**What you need:**
1. A T4 (free Colab) or better GPU runtime. A100 strongly recommended
   for `K_max=4` (sequence length = `(K+1)·L = 980` tokens).
2. A single zip file uploaded to your Google Drive containing the
   `p10/, p11/, …, p19/` patient folders — i.e. the standard
   MIMIC-CXR-JPG layout: `pXX/p{subject_id}/s{study_id}/{dicom_id}.jpg`.
   That's it — no other files needed. The CSVs (`subset_out/*.csv`) and
   the upstream BioViL-T weights are already in the repo / auto-
   downloaded by hi-ml respectively.

**Order of operations:**
- Section 1: clone + install + verify GPU
- Section 2: mount Drive, unzip data to /content (local SSD is much
  faster than Drive for the inner training loop)
- Section 3: launch single-GPU training with overridden paths
- Section 4: copy checkpoints back to Drive (optional)

## 1. Setup

In [ ]:
# Clone the repo and enter it (idempotent - safe to re-run after a start)
import os
if not os.path.exists('/content/cxr-temporal-multiprior'):
    !git clone https://github.com/nprakash1/cxr-temporal-multiprior.git /content/cxr-temporal-multiprior
else:
    print('Repo already cloned - pulling latest...')
    !cd /content/cxr-temporal-multiprior && git pull --ff-only
%cd /content/cxr-temporal-multiprior

In [ ]:
# Install dependencies (~2-3 min) + auto-restart runtime
# ----------------------------------------------------------------
# requirements.txt pins numpy<2.0 (required by health_multimodal).
# Colab ships numpy 2.x, so this is a DOWNGRADE. numpy's compiled C
# extension CANNOT be hot-swapped in a live kernel, so we must restart
# the runtime after installing - otherwise you get:
#   AttributeError: module 'numpy._core._multiarray_umath' has no
#   attribute '_blas_supports_fpe'
#
# A sentinel file prevents a re-install/re-restart loop. After the
# runtime restarts, just re-run cell 2 (it only re-cds) and then re-run
# THIS cell - it will detect the sentinel and skip straight through.
# ----------------------------------------------------------------
import os

SENTINEL = '/content/.deps_installed'

if not os.path.exists(SENTINEL):
    !pip install -q -r requirements.txt
    !pip install -q hi-ml-multimodal
    open(SENTINEL, 'w').close()
    print('\nInstall complete. Restarting runtime so the correct numpy loads...')
    print('After restart: re-run cell 2, then re-run THIS cell (it will skip).')
    os.kill(os.getpid(), 9)   # force a clean kernel restart
else:
    print('Deps already installed (sentinel found) - runtime restarted cleanly.')
    import numpy, scipy
    print(f'   numpy {numpy.__version__}  .  scipy {scipy.__version__}')

In [ ]:
# GPU check
import torch
assert torch.cuda.is_available(), 'No GPU detected. Runtime → Change runtime type → GPU.'
print('device :', torch.cuda.get_device_name(0))
print('mem    :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')

## 2. Mount Drive and unzip your MIMIC subset

**Edit `ZIP_PATH` below** to point at your uploaded zip on Drive.
Expected zip layout after extraction:

```
<wherever it lands>/
├── p10/
│   └── p10054639/
│       └── s55344325/
│           └── 2cceb6b7-….jpg
├── p11/
├── …
└── p19/
```

If your zip extracts to `mimic_subset/p10/...`, set
`IMAGE_ROOT = '/content/mimic_subset'`. If it extracts to a `files/`
subdir, point at that instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# >>> EDIT THIS to your actual zip path on Drive <<<
ZIP_PATH    = '/content/drive/MyDrive/mimic_subset.zip'
EXTRACT_DIR = '/content'                      # local SSD; fastest for training

import os, time
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}. Fix ZIP_PATH above.'
print(f'Zip size: {os.path.getsize(ZIP_PATH) / 1024**3:.2f} GiB')

t0 = time.time()
!unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"
print(f'Unzip done in {time.time()-t0:.0f}s')

In [ ]:
# Auto-detect the IMAGE_ROOT (the dir containing pXX/ folders).
import os, glob
candidates = []
for root, dirs, _ in os.walk(EXTRACT_DIR):
    # IMAGE_ROOT is the dir whose direct children are pXX/.
    if any(d.startswith('p') and len(d) == 3 and d[1:].isdigit() for d in dirs):
        candidates.append(root)
        break

assert candidates, (
    f'Could not find a pXX/ layout under {EXTRACT_DIR}. '
    f'List the dir manually: !ls {EXTRACT_DIR}'
)
IMAGE_ROOT = candidates[0]
print(f'IMAGE_ROOT = {IMAGE_ROOT}')
print('top-level patient dirs:', sorted(
    d for d in os.listdir(IMAGE_ROOT) if d.startswith('p') and len(d) == 3
)[:15])

In [ ]:
# Quick sanity check: can we resolve a sample image from the train CSV?
import pandas as pd, os
df = pd.read_csv('subset_out/biovilt_pretrain_train_imagelevel.csv')
print(f'rows: {len(df)}  splits: {df["split"].unique()}')
row = df.iloc[0]
pid = str(row['subject_id'])
path = os.path.join(
    IMAGE_ROOT,
    f'p{pid[:2]}', f'p{pid}',
    f's{row["study_id"]}',
    f'{row["dicom_id_curr"]}.jpg',
)
print('sample path:', path)
print('exists?    :', os.path.exists(path))
assert os.path.exists(path), (
    'First sample image not found — your zip layout may differ. '
    'Inspect IMAGE_ROOT manually and update if needed.'
)

## 2a. Batch disease-entropy check (data quality · no training)

Read-only diagnostic. Joins the official **CheXpert** label CSV to your
train split so every image gets its 14-disease "fingerprint", then
simulates batches at `BATCH_SIZE` and reports two numbers:

- **`profile_entropy_norm`** (0–1, want **~1.0**) — how diverse the
  disease fingerprints are within a batch.
- **`collision_rate`** (0–1, want **~0**) — fraction of images that
  share an identical fingerprint with another image in the same batch.

Low entropy / high collisions ⇒ many identical fingerprints per batch
(usually lots of `No Finding` normals). Those look the same to the model
but the contrastive loss pushes them apart anyway → false negatives →
the kind of noisy signal behind a drifting global-contrastive val loss.
If the numbers are bad, the fix is a disease-aware batch sampler.

**No edits to training/smoke-test/model cells** — pure measurement.
Run it before training so a bad subset costs seconds, not 80 minutes.


In [ ]:
# >>> DISEASE-ENTROPY DIAGNOSTIC <<<
# Read-only data-quality check. Does NOT train, does NOT touch the model.
# ----------------------------------------------------------------------
import os, math, numpy as np, pandas as pd

# 1) CheXpert labels. Default = the filtered subset file committed to the
#    repo (built locally once via biovilt/make_chexpert_subset.py), so it
#    ships with the git clone — no Drive upload or PhysioNet creds needed.
#    Columns: subject_id, study_id, + 14 disease columns
#    (1=present, 0=absent, -1=uncertain, blank=not mentioned).
CHEXPERT_CSV = 'subset_out/chexpert_labels_subset.csv'   # repo-committed (Option A)
# If you didn't commit it, fall back to a Drive upload of the full sheet:
# CHEXPERT_CSV = '/content/drive/MyDrive/mimic-cxr-2.0.0-chexpert.csv'


# Batch size to simulate. Falls back to 32 if you run this before cell 11.
BATCH_SIZE_DIAG = int(globals().get('BATCH_SIZE', 32))
N_SIM_BATCHES   = 200          # how many random batches to average over
SEED            = 0

assert os.path.exists(CHEXPERT_CSV), (
    f'CheXpert CSV not found at {CHEXPERT_CSV}. Upload it to Drive and fix '
    f'CHEXPERT_CSV above.'
)

# 2) Load both tables and align the join key types (silent-merge guard).
train = pd.read_csv('subset_out/biovilt_pretrain_train_imagelevel.csv')
chex  = pd.read_csv(CHEXPERT_CSV)
train['study_id'] = train['study_id'].astype('int64')
chex['study_id']  = chex['study_id'].astype('int64')

# 3) Identify the 14 disease columns (everything that isn't an id column).
id_cols      = {'subject_id', 'study_id'}
disease_cols = [c for c in chex.columns if c not in id_cols]
print(f'disease columns ({len(disease_cols)}): {disease_cols}')

# 4) Join: every training row gets its study's 14-disease fingerprint.
merged = train.merge(chex[['study_id'] + disease_cols], on='study_id', how='left')
matched = merged[disease_cols].notna().any(axis=1).mean()
print(f'rows: {len(merged)}   labels matched: {matched:5.1%}')
assert matched > 0, 'No study_id matched — check the CSV is the right CheXpert sheet.'

# 5) Build a binary "present" fingerprint per row (present=1, else 0),
#    then a compact string id so identical fingerprints compare equal.
present = (merged[disease_cols].fillna(0.0).values == 1.0).astype(np.int8)
fingerprints = np.array([''.join(map(str, row)) for row in present])

# 6) Simulate random batches and average the two metrics.
def batch_metrics(labels, B):
    n = len(labels)
    _, counts = np.unique(labels, return_counts=True)
    p = counts / n
    H = -(p * np.log(p)).sum()
    ent_norm = H / math.log(B) if B > 1 else 0.0          # normalize by max possible
    collisions = (counts[counts > 1].sum()) / n           # share-a-fingerprint frac
    return ent_norm, collisions

rng = np.random.default_rng(SEED)
ent_list, col_list = [], []
idx = np.arange(len(fingerprints))
for _ in range(N_SIM_BATCHES):
    pick = rng.choice(idx, size=min(BATCH_SIZE_DIAG, len(idx)), replace=False)
    e, c = batch_metrics(fingerprints[pick], BATCH_SIZE_DIAG)
    ent_list.append(e); col_list.append(c)

profile_entropy_norm = float(np.mean(ent_list))
collision_rate       = float(np.mean(col_list))

# 7) Context: what fraction of the whole subset is the single most common
#    fingerprint (usually all-zeros = "No Finding")?
uniq, cnts = np.unique(fingerprints, return_counts=True)
top_frac = cnts.max() / cnts.sum()

print('\n================ BATCH DISEASE-ENTROPY ================')
print(f'  batch size simulated : {BATCH_SIZE_DIAG}  (x{N_SIM_BATCHES} batches)')
print(f'  profile_entropy_norm : {profile_entropy_norm:5.3f}   (want ~1.0)')
print(f'  collision_rate       : {collision_rate:5.3f}   (want ~0.0)')
print(f'  most-common profile  : {top_frac:5.1%} of all rows')
print('======================================================')
if profile_entropy_norm < 0.85 or collision_rate > 0.15:
    print('  ⚠️  Batches collide on identical fingerprints — likely a chunk of')
    print('      "No Finding" normals. Consider a disease-aware batch sampler')
    print('      before trusting the global-contrastive loss.')
else:
    print('  ✅  Batches look diverse. Global-contrastive signal should be clean.')


## 3. Train

`resume_train.py` reads `IMAGE_ROOT`, `CSV_DIR`, `CHECKPOINT_DIR`,
`LOG_DIR`, and `NUM_WORKERS` from env vars (or equivalent CLI flags),
and uses `--epochs` / `--batch-size` to override defaults for smoke
runs. We launch with `torchrun --nproc_per_node=1` because Colab gives
us a single GPU; the DDP code path still works fine at world_size=1.

**Memory note for K_max:**
- `K_max=1`: ~2L = 392 tokens in the self-attn. Fits easily on T4 (16GB).
- `K_max=4`: 5L = 980 tokens. Needs A100 (40GB+) or batch_size=4 on T4.

Adjust `BATCH_SIZE` below if you OOM.

In [ ]:
# Training config — edit freely
K_MAX      = 4              # 1 = reproduce upstream BioViL-T (2-config ablation only)
                            # 4 = full multi-prior; gives 5-point Family A history curve.
                            # This is the actual experiment your project is designed for.
BATCH_SIZE = 32             # A100 40GB at K_max=4: 32 is comfortable (~15-20 GB VRAM).
                            #   try 48 for stronger contrastive signal if VRAM allows.
                            # On T4 16GB: K_max=1→16-32; K_max=4→4 (will be very slow).
EPOCHS     = 50             # full BioViL-T-style pretraining schedule
MODE       = 'biovilt'      # 'biovil' | 'biovilt' | 'biovilt_finetuned'
                            # NOTE: we do *not* pass --init-from or --resume, so the
                            # image encoder uses whatever weights `mode` constructs
                            # (auto-downloaded by hi-ml) and the multi-prior pooler /
                            # projection heads / text heads are all freshly initialized.
                            # This is the closest you can get to 'BioViL-T from scratch'
                            # without retraining a CNN from random init.

import os
os.environ['IMAGE_ROOT']     = IMAGE_ROOT
os.environ['CSV_DIR']        = '/content/cxr-temporal-multiprior/subset_out'
os.environ['CHECKPOINT_DIR'] = '/content/checkpoints'
os.environ['LOG_DIR']        = '/content/logs'
os.environ['NUM_WORKERS']    = '8'   # A100 high-RAM has ~12 vCPU; use 8 to keep
                                     # the GPU fed without saturating the CPU.
                                     # On free Colab (T4, 2 vCPU): set this to 2.
os.environ['K_MAX']          = str(K_MAX)     # exposed for Section 6 ablation cells
os.environ['MODE']           = MODE
os.environ['BATCH_SIZE']     = str(BATCH_SIZE)

# A100 throughput hint: let cuDNN pick the fastest conv algos after warmup.
# (No-op if your training script doesn't use cuDNN benchmark.)
os.environ['TORCH_CUDNN_BENCHMARK'] = '1'


In [ ]:
# The training CSV references both train and val splits via the `split` column,
# but resume_train.py looks for a *_combined_imagelevel.csv for val. The repo
# ships train/val/test separately. Easiest fix: symlink val→combined.
import os, shutil
src = '/content/cxr-temporal-multiprior/subset_out/biovilt_pretrain_val_imagelevel.csv'
dst = '/content/cxr-temporal-multiprior/subset_out/biovilt_pretrain_combined_imagelevel.csv'
if not os.path.exists(dst):
    shutil.copyfile(src, dst)
    print('Created combined CSV (copy of val).')
else:
    print('Combined CSV already exists.')

## 3a. Smoke test — verify multi-prior architecture (run before training)

Runs 4 forward-pass scenarios on synthetic random data to verify the multi-prior changes work.
Catches shape bugs, padding-mask leaks, K=0 fallback bugs, and migration errors **before** you spend 80 min training.

Takes ~3-10 seconds on GPU. If anything fails it raises AssertionError with the exact mismatch.

In [ ]:
# 🧪 SMOKE TEST — verify multi-prior architecture correctness
# ============================================================
# This cell runs 4 forward-pass scenarios on SYNTHETIC random data to
# verify that the multi-prior changes work. It does NOT do any training.
# Run time: ~3-10 seconds on GPU, ~30 seconds on CPU.
#
# If anything is broken, this cell will fail with AssertionError BEFORE
# you spend 80 minutes training a bad model.
#
# Tests:
#   Section 4 — Temporal embedding migration (init correctness)
#   Section 5 — Scenario A: K=4 batch (all priors present)
#   Section 6 — Scenario B: K=0 batch (no priors -> fast path)
#   Section 7 — Scenario C: Mixed batch [K=0, K=2, K=4]
#   Section 8 — Scenario D: Padding isolation (the hardest case)

import os, sys, contextlib, io
import torch
import torch.nn.functional as F

# Make sure the repo is on the path
REPO_DIR = "/content/cxr-temporal-multiprior"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "biovilt"))

from biovilt.tempcxr.modules.tempcxr_model import TempCXR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32   # smoke test uses fp32 for exact-equality checks

# Color helpers (works in Colab + terminal)
GREEN  = "\033[92m"
RED    = "\033[91m"
BOLD   = "\033[1m"
END    = "\033[0m"

def section(title):
    bar = "─" * 64
    print(f"\n{bar}\n{BOLD}{title}{END}\n{bar}")

def check(label, condition, detail=""):
    tag = f"{GREEN}✅{END}" if condition else f"{RED}❌{END}"
    suffix = f"  ({detail})" if detail else ""
    print(f"  {tag}  {label}{suffix}")
    if not condition:
        raise AssertionError(label)

# ============================================================
# SECTION 1: BUILD MODEL + REGISTER FORWARD HOOKS
# ============================================================
print(f"{BOLD}══════════════════════════════════════════════════════════════{END}")
print(f"{BOLD} 🧪 SMOKE TEST  ·  TempCXR multi-prior architecture           {END}")
print(f"{BOLD}══════════════════════════════════════════════════════════════{END}")
print(f"Device : {DEVICE}")
print(f"K_max  : 4  ·  B=3  ·  using synthetic random inputs (NOT real X-rays)")

# Build model
print("\nBuilding model (downloads ~441MB BioViL-T weights if not cached)…")
torch.manual_seed(0)
with contextlib.redirect_stdout(io.StringIO()):  # silence model's own init prints
    model = TempCXR(mode="biovilt", K_max=4).to(DEVICE).to(DTYPE).eval()

# Register forward hooks on key modules
hook_handles = []
hook_log = []  # captured shapes per forward pass

def make_hook(name):
    def hook(module, inputs, output):
        in_shapes = []
        for x in inputs:
            if isinstance(x, torch.Tensor):
                in_shapes.append(tuple(x.shape))
        if isinstance(output, torch.Tensor):
            out_shape = tuple(output.shape)
        elif isinstance(output, (tuple, list)) and len(output) and isinstance(output[0], torch.Tensor):
            out_shape = tuple(output[0].shape)
        else:
            out_shape = "non-tensor"
        hook_log.append((name, in_shapes, out_shape))
    return hook

ie = model.image_encoder
hook_handles.append(ie.model.encoder.encoder.register_forward_hook(make_hook("ResNet50_trunk")))
hook_handles.append(ie.model.encoder.backbone_to_vit.register_forward_hook(make_hook("backbone_to_vit (1x1 conv)")))
hook_handles.append(ie.multi_pooler.register_forward_hook(make_hook("MultiPriorTransformerPooler")))

# ============================================================
# SECTION 2-3: Helper functions to build synthetic batches
# ============================================================
def make_inputs(num_priors_per_sample, seed=0):
    """Build curr_imgs, prior_imgs, prior_mask for a batch of B samples
    where sample i has num_priors_per_sample[i] real priors."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    B = len(num_priors_per_sample)
    K_max_local = max(num_priors_per_sample) if num_priors_per_sample else 0
    curr = torch.randn(B, 3, 448, 448, generator=g).to(DEVICE).to(DTYPE)
    if K_max_local == 0:
        return curr, None, None
    priors = torch.randn(B, K_max_local, 3, 448, 448, generator=g).to(DEVICE).to(DTYPE)
    mask = torch.zeros(B, K_max_local, dtype=torch.bool, device=DEVICE)
    for i, k_i in enumerate(num_priors_per_sample):
        mask[i, :k_i] = True
    return curr, priors, mask

texts = ["fake report A", "fake report B", "fake report C"]

# ============================================================
# SECTION 4: MIGRATION CHECK (no forward, inspect weights)
# ============================================================
section("Section 4: Temporal embedding migration (init correctness)")

te_multi = ie.multi_pooler.type_embed_multi.data
te_upstream = ie.multi_pooler.upstream.type_embed.data

check(
    "type_embed_multi shape = (K_max+1, 1, D) = (5, 1, 256)",
    te_multi.shape == (5, 1, 256),
    detail=f"got {tuple(te_multi.shape)}",
)
check(
    "row 0 ≡ upstream curr row",
    torch.allclose(te_multi[0], te_upstream[0]),
)
for k in range(1, 5):
    check(
        f"row {k} ≡ upstream prior row (copy+replicate)",
        torch.allclose(te_multi[k], te_upstream[1]),
    )
check("Migration init correct", True)

# ============================================================
# SECTION 5: SCENARIO A — K=4 batch (all priors)
# ============================================================
section("Section 5: Scenario A — K=4 batch (all samples have 4 priors)")
hook_log.clear()
curr_A, priors_A, mask_A = make_inputs([4, 4, 4], seed=42)

with torch.no_grad():
    out_A = model(curr_A, priors_A, mask_A, texts=texts)

print(f"  Forward pass hook trace:")
for name, in_sh, out_sh in hook_log:
    print(f"    [{name:38s}]  in={in_sh}  →  out={out_sh}")

check(
    "img_global.shape == (3, 128)",
    out_A["img_global"].shape == (3, 128),
    detail=f"got {tuple(out_A['img_global'].shape)}",
)
check(
    "img_patches.shape == (3, 196, 128)",
    out_A["img_patches"].shape == (3, 196, 128),
    detail=f"got {tuple(out_A['img_patches'].shape)}",
)
norms = out_A["img_global"].norm(dim=-1)
check(
    "||img_global[i]|| ≈ 1 (L2-normalized)",
    torch.allclose(norms, torch.ones_like(norms), atol=1e-4),
    detail=f"norms = {norms.tolist()}",
)
check("Scenario A (K=4) correct", True)

# ============================================================
# SECTION 6: SCENARIO B — K=0 batch (no priors, fast path)
# ============================================================
section("Section 6: Scenario B — K=0 batch (no priors → fast path)")
hook_log.clear()
curr_B, priors_B, mask_B = make_inputs([0, 0, 0], seed=42)

with torch.no_grad():
    out_B = model(curr_B, priors_B, mask_B, texts=texts)

print(f"  Forward pass hook trace:")
for name, in_sh, out_sh in hook_log:
    print(f"    [{name:38s}]  in={in_sh}  →  out={out_sh}")

pooler_fired = any(name == "MultiPriorTransformerPooler" for name, _, _ in hook_log)
check(
    "MultiPriorTransformerPooler SKIPPED (fast path via missing_previous_emb)",
    not pooler_fired,
    detail="pooler should not be invoked when no priors anywhere",
)
check(
    "img_global.shape == (3, 128)",
    out_B["img_global"].shape == (3, 128),
)
check(
    "img_patches.shape == (3, 196, 128)",
    out_B["img_patches"].shape == (3, 196, 128),
)

delta_AB = (out_A["img_global"] - out_B["img_global"]).abs().mean().item()
check(
    "Scenario A (with priors) ≠ Scenario B (no priors)",
    delta_AB > 1e-4,
    detail=f"mean |Δ| = {delta_AB:.4e}",
)
check("Scenario B (K=0 fast path) correct", True)

# ============================================================
# SECTION 7: SCENARIO C — Mixed batch [K=0, K=2, K=4]
# ============================================================
section("Section 7: Scenario C — Mixed batch [K=0, K=2, K=4]")
hook_log.clear()

curr_C, priors_C, mask_C = make_inputs([0, 2, 4], seed=42)

with torch.no_grad():
    out_C = model(curr_C, priors_C, mask_C, texts=texts)

print(f"  Forward pass hook trace:")
for name, in_sh, out_sh in hook_log:
    print(f"    [{name:38s}]  in={in_sh}  →  out={out_sh}")

check(
    "img_global.shape == (3, 128)",
    out_C["img_global"].shape == (3, 128),
)
delta_C0_B0 = (out_C["img_global"][0] - out_B["img_global"][0]).abs().max().item()
check(
    "Mixed batch sample 0 (K=0) ≡ Scenario B sample 0 (K=0)",
    delta_C0_B0 < 1e-4,
    detail=f"max |Δ| = {delta_C0_B0:.2e}",
)
delta_C2_A2 = (out_C["img_global"][2] - out_A["img_global"][2]).abs().max().item()
check(
    "Mixed batch sample 2 (K=4) ≡ Scenario A sample 2 (K=4)",
    delta_C2_A2 < 1e-3,
    detail=f"max |Δ| = {delta_C2_A2:.2e} (small numerical drift is OK)",
)
check("Scenario C (mixed batch) correct", True)

# ============================================================
# SECTION 8: SCENARIO D — Padding isolation stress test
# ============================================================
section("Section 8: Scenario D — Padding isolation stress test")

priors_D = priors_C.clone()
torch.manual_seed(999)
priors_D[1, 2:4] = torch.randn_like(priors_D[1, 2:4]) * 100.0

with torch.no_grad():
    out_D = model(curr_C, priors_D, mask_C, texts=texts)

delta_D1 = (out_D["img_global"][1] - out_C["img_global"][1]).abs().max().item()
check(
    "Sample 1 output bit-identical after perturbing padded slots",
    delta_D1 < 1e-4,
    detail=f"max |Δ| = {delta_D1:.2e}  (should be ~0; padded slots must NOT leak)",
)
delta_D2 = (out_D["img_global"][2] - out_C["img_global"][2]).abs().max().item()
check(
    "Sample 2 output unchanged (its slots not perturbed)",
    delta_D2 < 1e-4,
    detail=f"max |Δ| = {delta_D2:.2e}",
)
check("Scenario D (padding isolation) correct", True)

# ============================================================
# WRAP UP
# ============================================================
for h in hook_handles:
    h.remove()

print(f"\n{BOLD}══════════════════════════════════════════════════════════════{END}")
print(f"{BOLD}{GREEN} ALL CHECKS PASSED ✅  ·  Multi-prior architecture verified  {END}")
print(f"{BOLD}══════════════════════════════════════════════════════════════{END}")
print(f"Safe to proceed with training in the next cell.\n")


In [ ]:
# Launch training. torchrun handles the LOCAL_RANK env var that DDP needs.
!cd /content/cxr-temporal-multiprior && \
    torchrun --nproc_per_node=1 --master_port=29501 \
        biovilt/resume_train.py \
        --k-max {K_MAX} \
        --mode  {MODE} \
        --batch-size {BATCH_SIZE} \
        --epochs {EPOCHS}

## 4. Plot the loss curves

After training, `val_metrics.csv` has per-epoch validation losses
(total + global contrastive + local contrastive + MLM). Below also
shows train loss by parsing tqdm output stored alongside.

Published reference (BioViL-T, MIMIC-CXR full train set, ~400k pairs):
- Final global contrastive loss ≈ 1.4–1.7 on val
- Final local contrastive loss ≈ 0.6–0.9
- Final MLM loss ≈ 1.1–1.4

On a 500-patient subset you should expect somewhat higher final
values (less data → less effective contrastive temperature), but a
monotonically decreasing curve on all four signals is the right
qualitative result.

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt

csv_path = '/content/logs/val_metrics.csv'
assert os.path.exists(csv_path), f'{csv_path} missing — training may have crashed'

df = pd.read_csv(csv_path)
print(df.tail(10).to_string(index=False))
print(f'\nbest epoch (by val_total): {int(df.loc[df.val_total.idxmin(), "epoch"])} '
      f'  val_total = {df.val_total.min():.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)

axes[0, 0].plot(df.epoch, df.val_total,  marker='o', color='black')
axes[0, 0].set_title('Validation — total loss')
axes[0, 0].set_ylabel('loss')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(df.epoch, df.val_global, marker='o', color='tab:blue')
axes[0, 1].set_title('Validation — global contrastive')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(df.epoch, df.val_local,  marker='o', color='tab:orange')
axes[1, 0].set_title('Validation — local contrastive')
axes[1, 0].set_xlabel('epoch')
axes[1, 0].set_ylabel('loss')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(df.epoch, df.val_mlm,    marker='o', color='tab:green')
axes[1, 1].set_title('Validation — MLM')
axes[1, 1].set_xlabel('epoch')
axes[1, 1].grid(alpha=0.3)

fig.suptitle(f'cxr-temporal-multiprior  |  K_max={K_MAX}  mode={MODE}  '
             f'bs={BATCH_SIZE}  epochs={EPOCHS}', fontsize=11)
fig.tight_layout()
out = '/content/logs/loss_curves.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Saved {out}')
plt.show()

In [ ]:
# Overlay plot — all four val losses on one normalized axis, useful for
# spotting which signal dominates / saturates first.
fig, ax = plt.subplots(figsize=(9, 5))
for col, label, color in [
    ('val_total',  'Total',             'black'),
    ('val_global', 'Global contrastive','tab:blue'),
    ('val_local',  'Local contrastive', 'tab:orange'),
    ('val_mlm',    'MLM',               'tab:green'),
]:
    series = df[col]
    norm   = (series - series.min()) / (series.max() - series.min() + 1e-12)
    ax.plot(df.epoch, norm, marker='o', label=label, color=color, lw=1.5)

ax.set_xlabel('epoch')
ax.set_ylabel('min-max normalized loss')
ax.set_title(f'Validation losses (normalized) — K_max={K_MAX}')
ax.grid(alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig('/content/logs/loss_curves_normalized.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Save checkpoints back to Drive (optional)

Colab wipes `/content` between sessions, so persist your trained
weights to Drive if you want to resume later.

In [ ]:
!mkdir -p /content/drive/MyDrive/cxr_temporal_ckpts
!cp /content/checkpoints/best.pt              /content/drive/MyDrive/cxr_temporal_ckpts/best_kmax{K_MAX}.pt
!cp /content/logs/val_metrics.csv             /content/drive/MyDrive/cxr_temporal_ckpts/val_metrics_kmax{K_MAX}.csv
!cp /content/logs/loss_curves.png             /content/drive/MyDrive/cxr_temporal_ckpts/loss_curves_kmax{K_MAX}.png
!cp /content/logs/loss_curves_normalized.png  /content/drive/MyDrive/cxr_temporal_ckpts/loss_curves_norm_kmax{K_MAX}.png
!ls -lh /content/drive/MyDrive/cxr_temporal_ckpts

## 6. Family A ablation — does each additional prior help?

After training, this section evaluates the model under **incremental
history ablation** without any retraining:

- `history_0`: current image only (no priors used)
- `history_1`: current + 1 most-recent prior (= BioViL-T K=1 baseline)
- `history_2`: current + 2 most-recent priors
- ... up to `history_K_MAX`.

For each config we override `prior_mask` so that only the first N prior
slots are kept (`mask[:, N:] = False`). The model already supports this
natively — no architecture change needed. With your current `K_MAX=1`
this gives a 2-point comparison (history_0 vs history_1); rerunning
with `K_MAX=4` would give a 5-point history-length curve.

**Why this is fair without retraining:** real MIMIC patients naturally
have variable history lengths (0–4 priors), so the dataset already
trains the model on every contiguous-from-slot-0 mask pattern. Eval-time
truncation of trailing slots is in-distribution.


In [ ]:
# 6a. Load the best checkpoint and rebuild the val loader.
import os, sys, glob, torch, pandas as pd
from torch.utils.data import DataLoader

REPO = '/content/cxr-temporal-multiprior'
sys.path.insert(0, os.path.join(REPO, 'biovilt'))

from dataset import BioViLTDataset, biovilt_collate_fn
from tempcxr.modules.tempcxr_model import TempCXR
from losses import global_contrastive_loss, local_contrastive_loss, mlm_loss

# Same hyperparams used in training
K_MAX        = int(os.environ.get('K_MAX', 1))
MODE         = os.environ.get('MODE', 'biovilt')
BATCH_SIZE   = int(os.environ.get('BATCH_SIZE', 16))
IMAGE_ROOT   = os.environ['IMAGE_ROOT']
VAL_CSV      = os.path.join(REPO, 'subset_out', 'biovilt_pretrain_combined_imagelevel.csv')
CKPT_PATH    = '/content/checkpoints/best.pt'

# --- Model ---
device = 'cuda'
model  = TempCXR(mode=MODE, K_max=K_MAX).to(device).eval()
ckpt   = torch.load(CKPT_PATH, map_location=device)
state  = ckpt.get('model', ckpt)
state  = {k.replace('module.', ''): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
print(f'Loaded {CKPT_PATH}  (missing={len(missing)}, unexpected={len(unexpected)})')

# --- Val loader (no DistributedMixedBatchSampler needed for single-process eval) ---
val_ds = BioViLTDataset(
    csv_path=VAL_CSV,
    image_root=IMAGE_ROOT,
    split='val',
    train=False,
    k_max=K_MAX,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, collate_fn=biovilt_collate_fn, pin_memory=True,
)
print(f'Val samples: {len(val_ds)}  |  batches: {len(val_loader)}  |  K_MAX={K_MAX}')


In [ ]:
# 6b. Run all Family A configs and report a table + bar chart.
import torch, pandas as pd, matplotlib.pyplot as plt

@torch.no_grad()
def evaluate(n_priors_kept):
    model.eval()
    sums = dict(total=0.0, g=0.0, l=0.0, m=0.0)
    n = 0
    for batch in val_loader:
        curr       = batch['current_image'].to(device)
        prior_imgs = batch['prior_images']
        prior_mask = batch['prior_mask']
        texts      = batch['text']
        if prior_imgs is not None:
            prior_imgs = prior_imgs.to(device)
            prior_mask = prior_mask.to(device).clone()
            # Family A: keep only the n_priors_kept most-recent prior slots
            if n_priors_kept < prior_mask.shape[1]:
                prior_mask[:, n_priors_kept:] = False
        with torch.amp.autocast('cuda'):
            out = model(curr, prior_imgs, prior_mask, texts=texts)
            loss_g = global_contrastive_loss(out['img_global'],   out['txt_global'])
            loss_l = local_contrastive_loss (out['img_patches'],  out['txt_local'], out['token_mask'])
            loss_m = mlm_loss              (out['mlm_logits'],   out['mlm_labels'])
            loss   = loss_g + loss_l + loss_m
        b = curr.shape[0]
        sums['total'] += loss.item()   * b
        sums['g']     += loss_g.item() * b
        sums['l']     += loss_l.item() * b
        sums['m']     += loss_m.item() * b
        n += b
    return {k: v / max(n, 1) for k, v in sums.items()}

rows = []
for n in range(K_MAX + 1):
    print(f'Evaluating history_{n} ...', end=' ', flush=True)
    r = evaluate(n)
    rows.append({'config': f'history_{n}', 'priors_kept': n, **r})
    print(f"total={r['total']:.4f}  g={r['g']:.4f}  l={r['l']:.4f}  mlm={r['m']:.4f}")

df = pd.DataFrame(rows)
print('\nFamily A — Incremental history ablation:')
print(df.to_string(index=False))

# Save and plot
out_csv = '/content/logs/family_a_ablation.csv'
df.to_csv(out_csv, index=False)
print(f'\nSaved to {out_csv}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(df['config'], df['total'], color='steelblue')
ax[0].set_ylabel('Val total loss'); ax[0].set_title('Family A — total loss')
for x, y in zip(df['config'], df['total']):
    ax[0].text(x, y, f'{y:.3f}', ha='center', va='bottom', fontsize=9)

ax[1].plot(df['priors_kept'], df['m'], 'o-', label='MLM',    color='tab:red')
ax[1].plot(df['priors_kept'], df['g'], 's-', label='global', color='tab:blue')
ax[1].plot(df['priors_kept'], df['l'], '^-', label='local',  color='tab:green')
ax[1].set_xlabel('# priors kept'); ax[1].set_ylabel('Val loss')
ax[1].set_title('Per-loss breakdown'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Troubleshooting

**`AssertionError: first sample image not found`** — your zip's
directory structure doesn't have `pXX/` directly under the auto-
detected `IMAGE_ROOT`. Run `!find /content -maxdepth 4 -type d -name 'p1*' | head` to
see the actual layout, then set `IMAGE_ROOT` manually and re-run.

**`CUDA out of memory`** — drop `BATCH_SIZE` to 4 or 2, or use
`K_MAX=2` instead of 4. Free-tier T4 has only 16GB. If still OOM,
Colab Pro's A100 is the cleanest fix.

**`NCCL error`** — should be rare on Colab but if it happens, you can
fall back to gloo by editing the `setup_ddp` call in
`biovilt/resume_train.py` from `nccl` to `gloo`. Single-GPU
performance is identical.

**Training is slow** — the bottleneck is JPEG decoding through 2
DataLoader workers. Set `NUM_WORKERS=4` if you have Colab Pro with
more vCPUs.
